In [1]:
import os
import pickle
import functools
import numpy as np
import pandas as pd

In [2]:
fs = ["linear", "quadratic", "step", "piecewiselinear"]
ns = [11, 31]
methods = ["TabPFN", "GP"]
save_folder = "output_coverage/raw_data"
nrep = 100

x_train = np.arange(-4.5, 4.6, 0.1)
interpolation_indices = (x_train <= 1) & (x_train >= -1)
extrapolation_indices = (x_train > 1) | (x_train < -1)


def cartesian_product_transpose(*arrays):
    """
    Transposes the cartesian product of the input arrays.
    """
    arrays = [np.asarray(arr)
              for arr in arrays]  # Ensure all inputs are arrays
    broadcastable = np.ix_(*arrays)
    broadcasted = np.broadcast_arrays(*broadcastable)

    # Simplified dtype assignment
    dtype = broadcasted[0].dtype

    rows, cols = functools.reduce(np.multiply,
                                  broadcasted[0].shape), len(broadcasted)
    out = np.empty(rows * cols, dtype=dtype)

    start, end = 0, rows
    for a in broadcasted:
        out[start:end] = a.reshape(-1)
        start, end = end, end + rows

    return out.reshape(cols, rows).T

In [3]:
iterpolation_coverage = np.zeros((nrep, len(fs), len(ns), len(methods)))
iterpolation_length = np.zeros((nrep, len(fs), len(ns), len(methods)))

extrapolation_coverage = np.zeros((nrep, len(fs), len(ns), len(methods)))
extrapolation_length = np.zeros((nrep, len(fs), len(ns), len(methods)))

for i in range(nrep):
    for j, func in enumerate(fs):
        for k, n in enumerate(ns):
            for l, method in enumerate(methods):
                save_file = os.path.join(
                    save_folder,
                    "case_" + str(i + 1) + "_func_" + func + "_n_" + str(n) +
                    ".pickle",
                )
                with open(save_file, "rb") as f:
                    output = pickle.load(f)
                iterpolation_coverage[i, j, k, l] = np.mean(output[method + "_cover"][interpolation_indices])
                iterpolation_length[i, j, k, l] = np.mean(
                    output[method + "_length"][interpolation_indices])
                extrapolation_coverage[i, j, k, l] = np.mean(
                    output[method + "_cover"][extrapolation_indices])
                extrapolation_length[i, j, k, l] = np.mean(
                    output[method + "_length"][extrapolation_indices])


### Interpolation

In [4]:
df = pd.DataFrame(
    cartesian_product_transpose(
        *[np.arange(i) for i in iterpolation_coverage.shape]),
    columns=["repetition", "Setup", "n", "Method"],
)

df["coverage"] = iterpolation_coverage.ravel()
df["n"] = df["n"].astype("category")
df["Setup"] = df["Setup"].astype("category")
df["Method"] = df["Method"].astype("category")

# rename the categories
df["n"] = df["n"].cat.rename_categories({0:"11", 1:"31"})
df["Setup"] = df["Setup"].cat.rename_categories({
    0: "linear",
    1: "quadratic",
    2: "step",
    3: "piecewiselinear"
})
df["Method"] = df["Method"].cat.rename_categories({0: "TabPFN", 1: "GP"})
iterpolation_coverage_mean = df.groupby(['Setup', 'n',
                                         'Method'])['coverage'].mean()
iterpolation_coverage_mean

/var/folders/jp/n0wjhx4j3v92_82rpzg2v00m0000gp/T/ipykernel_40594/316880853.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  iterpolation_coverage_mean = df.groupby(['Setup', 'n',


Setup            n   Method
linear           11  TabPFN    0.9350
                     GP        0.9455
                 31  TabPFN    0.9405
                     GP        0.8775
quadratic        11  TabPFN    0.9525
                     GP        0.9665
                 31  TabPFN    0.9420
                     GP        0.9390
step             11  TabPFN    0.9705
                     GP        0.8635
                 31  TabPFN    0.9365
                     GP        0.9050
piecewiselinear  11  TabPFN    0.9930
                     GP        0.9795
                 31  TabPFN    0.9375
                     GP        0.9550
Name: coverage, dtype: float64

In [5]:
df = pd.DataFrame(
    cartesian_product_transpose(
        *[np.arange(i) for i in iterpolation_length.shape]),
    columns=["repetition", "Setup", "n", "Method"],
)

df["coverage"] = iterpolation_length.ravel()
df["n"] = df["n"].astype("category")
df["Setup"] = df["Setup"].astype("category")
df["Method"] = df["Method"].astype("category")

# rename the categories
df["n"] = df["n"].cat.rename_categories({0: "11", 1: "31"})
df["Setup"] = df["Setup"].cat.rename_categories({
    0: "linear",
    1: "quadratic",
    2: "step",
    3: "piecewiselinear"
})
df["Method"] = df["Method"].cat.rename_categories({0: "TabPFN", 1: "GP"})
iterpolation_length_mean = df.groupby(['Setup', 'n',
                                       'Method'])['coverage'].mean()
iterpolation_length_mean

/var/folders/jp/n0wjhx4j3v92_82rpzg2v00m0000gp/T/ipykernel_40594/3100998822.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  iterpolation_length_mean = df.groupby(['Setup', 'n',


Setup            n   Method
linear           11  TabPFN    0.465589
                     GP        0.430352
                 31  TabPFN    0.392354
                     GP        0.326786
quadratic        11  TabPFN    0.736502
                     GP        0.543864
                 31  TabPFN    0.423516
                     GP        0.424692
step             11  TabPFN    1.230093
                     GP        0.693388
                 31  TabPFN    0.550136
                     GP        0.580771
piecewiselinear  11  TabPFN    1.844942
                     GP        0.697656
                 31  TabPFN    0.478826
                     GP        0.526800
Name: coverage, dtype: float64

## Extrapolation

In [6]:
df = pd.DataFrame(
    cartesian_product_transpose(
        *[np.arange(i) for i in extrapolation_coverage.shape]),
    columns=["repetition", "Setup", "n", "Method"],
)

df["coverage"] = extrapolation_coverage.ravel()
df["n"] = df["n"].astype("category")
df["Setup"] = df["Setup"].astype("category")
df["Method"] = df["Method"].astype("category")

# rename the categories
df["n"] = df["n"].cat.rename_categories({0: "11", 1: "31"})
df["Setup"] = df["Setup"].cat.rename_categories({
    0: "linear",
    1: "quadratic",
    2: "step",
    3: "piecewiselinear"
})
df["Method"] = df["Method"].cat.rename_categories({0: "TabPFN", 1: "GP"})
extrapolation_coverage_mean = df.groupby(['Setup', 'n',
                                          'Method'])['coverage'].mean()
extrapolation_coverage_mean

/var/folders/jp/n0wjhx4j3v92_82rpzg2v00m0000gp/T/ipykernel_40594/3009463415.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  extrapolation_coverage_mean = df.groupby(['Setup', 'n',


Setup            n   Method
linear           11  TabPFN    0.927606
                     GP        0.485634
                 31  TabPFN    0.982676
                     GP        0.617465
quadratic        11  TabPFN    0.428732
                     GP        0.079155
                 31  TabPFN    0.681690
                     GP        0.088169
step             11  TabPFN    0.999577
                     GP        0.999718
                 31  TabPFN    0.997465
                     GP        0.999859
piecewiselinear  11  TabPFN    1.000000
                     GP        1.000000
                 31  TabPFN    0.999296
                     GP        1.000000
Name: coverage, dtype: float64

In [7]:
df = pd.DataFrame(
    cartesian_product_transpose(
        *[np.arange(i) for i in extrapolation_length.shape]),
    columns=["repetition", "Setup", "n", "Method"],
)

df["coverage"] = extrapolation_length.ravel()
df["n"] = df["n"].astype("category")
df["Setup"] = df["Setup"].astype("category")
df["Method"] = df["Method"].astype("category")

# rename the categories
df["n"] = df["n"].cat.rename_categories({0: "11", 1: "31"})
df["Setup"] = df["Setup"].cat.rename_categories({
    0: "linear",
    1: "quadratic",
    2: "step",
    3: "piecewiselinear"
})
df["Method"] = df["Method"].cat.rename_categories({0: "TabPFN", 1: "GP"})
extrapolation_length_mean = df.groupby(['Setup', 'n',
                                        'Method'])['coverage'].mean()
extrapolation_length_mean

/var/folders/jp/n0wjhx4j3v92_82rpzg2v00m0000gp/T/ipykernel_40594/3733453393.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  extrapolation_length_mean = df.groupby(['Setup', 'n',


Setup            n   Method
linear           11  TabPFN     5.147827
                     GP         2.933313
                 31  TabPFN     5.924190
                     GP         2.613020
quadratic        11  TabPFN     8.474210
                     GP         2.731048
                 31  TabPFN    11.949167
                     GP         3.225911
step             11  TabPFN     3.369934
                     GP         2.406496
                 31  TabPFN     3.822717
                     GP         2.409613
piecewiselinear  11  TabPFN     2.474406
                     GP         1.776946
                 31  TabPFN     4.957968
                     GP         1.771025
Name: coverage, dtype: float64